# SVM — Linear (LinearSVC)

This notebook walks through the SVM — Linear (LinearSVC) pipeline end-to-end: data loading, feature construction, bounded group-CV tuning, validation evaluation, metrics, and interpretation.

## 1. Load prepared data and fixed splits

Loads the prepared development bundle through the project's shared splitter. Only train and validation partitions are used; the locked test partition is never materialized during development selection.

In [ ]:
import warnings
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
import sys
# Find project root (directory containing src/). Run from project root or
# nbconvert will still locate it via the marker file below.
import os
_cwd = Path.cwd()
_roots = [_cwd]
for _p in [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]:
    if (_p / "src" / "splitting.py").exists():
        _roots.insert(0, _p)
        break
ROOT = _roots[0]
sys.path.insert(0, str(ROOT))
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from src.splitting import load_development_bundle

LABELS = ["Safe", "Neutral", "Risky"]

# 1. Load prepared data and fixed splits (primary protocol; validation is held-out).
data, split_meta = load_development_bundle("primary", ROOT)
print("Split:", split_meta["split_id"])
print("Dataset SHA-256:", split_meta["dataset_sha256"])

train = data[data["primary_partition"].eq("train") & data["primary_include"]].copy().reset_index(drop=True)
validation = data[data["primary_partition"].eq("validation") & data["primary_include"]].copy().reset_index(drop=True)

print("Train rows:", len(train))
print("Validation rows:", len(validation))
print("Train label counts:")
print(train["label"].value_counts())
print("Validation label counts:")
print(validation["label"].value_counts())

# Group-respecting CV folds come from the frozen manifest (primary_cv_fold).
folds = sorted(train["primary_cv_fold"].unique().tolist())
print("CV folds:", folds)


## 2. Build the feature/model pipeline (sparse word unigram/bigram TF-IDF)

Builds the word unigram/bigram TF-IDF vectorizer fitted on training data only, then transforms train and validation. The matrix stays sparse — no dense conversion.

In [ ]:
# 2. Build the feature/model pipeline (sparse word unigram/bigram TF-IDF).
# Fitted on training data only; matrix stays sparse — no dense conversion.
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.99,
    max_features=60000, sublinear_tf=True, strip_accents="unicode",
    dtype=np.float32, lowercase=True,
)

X_train = vectorizer.fit_transform(train["quote_text"].fillna("").astype(str))
X_val = vectorizer.transform(validation["quote_text"].fillna("").astype(str))

print("TF-IDF train shape:", X_train.shape)
print("TF-IDF validation shape:", X_val.shape)
print("Dtype:", X_train.dtype)


In [ ]:
# 3. Train, bounded tune, and predict on validation.
# Uses the three fixed group-respecting CV folds; TF-IDF/SVD refit inside each fold.
from sklearn.svm import LinearSVC
MODEL_CLASS = lambda **p: LinearSVC(random_state=20261102, **p)
GRID = [{"C": c, "class_weight": cw, "max_iter": 5000} for c in (0.1, 1.0, 10.0) for cw in (None, "balanced")]

XTR = X_train
XVA = X_val
YTR = train["label"].to_numpy()
YVA = validation["label"].to_numpy()

fold_series = train["primary_cv_fold"].to_numpy()
best = None
fold_details = []
for params in GRID:
    fold_scores = []
    for fold in folds:
        fit_mask = fold_series != fold
        eval_mask = fold_series == fold
        model = MODEL_CLASS(**params)
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            model.fit(XTR[fit_mask], YTR[fit_mask])
        yp = model.predict(XTR[eval_mask])
        fold_scores.append(f1_score(YTR[eval_mask], yp, labels=LABELS, average="macro", zero_division=0))
    mean_score = float(np.mean(fold_scores))
    std_score = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0
    print(params, "-> CV macro F1:", round(mean_score, 4), "±", round(std_score, 4))
    fold_details.append({"params": params, "mean_cv": mean_score, "std_cv": std_score, "folds": fold_scores})
    if best is None or mean_score > best["mean_cv"]:
        best = {"params": params, "mean_cv": mean_score, "std_cv": std_score, "folds": fold_scores}

print()
print("Best params:", best["params"], "CV macro F1:", round(best["mean_cv"], 4), "±", round(best["std_cv"], 4))

final = MODEL_CLASS(**best["params"])
t0 = time.perf_counter()
final.fit(XTR, YTR)
fit_seconds = time.perf_counter() - t0

# Train performance (overfitting diagnosis — not used for selection).
y_train_pred = final.predict(XTR)
train_acc = float(accuracy_score(YTR, y_train_pred))
train_f1 = float(f1_score(YTR, y_train_pred, labels=LABELS, average="macro", zero_division=0))

t0 = time.perf_counter()
y_pred = final.predict(XVA)
predict_seconds = time.perf_counter() - t0

print("Fit seconds:", round(fit_seconds, 3))
print("Predict seconds:", round(predict_seconds, 3))
print("Train macro F1:", round(train_f1, 4))
print("Train accuracy:", round(train_acc, 4))


## 4. Calculate metrics on held-out validation

Computes accuracy, balanced accuracy, per-class precision/recall/F1/support, macro and weighted averages, Risky false positives/false negatives, and the confusion matrix.

In [ ]:
# 4. Calculate metrics on held-out validation.
report = classification_report(YVA, y_pred, labels=LABELS, output_dict=True, zero_division=0)

summary = {
    "rows": int(len(YVA)),
    "accuracy": float(accuracy_score(YVA, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(YVA, y_pred)),
    "macro_f1": float(f1_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "macro_precision": float(precision_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "risky_precision": float(precision_score(YVA, y_pred, labels=["Risky"], average="macro", zero_division=0)),
    "risky_recall": float(recall_score(YVA, y_pred, labels=["Risky"], average="macro", zero_division=0)),
    "risky_false_positives": int(((YVA != "Risky") & (y_pred == "Risky")).sum()),
    "risky_false_negatives": int(((YVA == "Risky") & (y_pred != "Risky")).sum()),
}
for label in LABELS:
    summary[f"{label.lower()}_precision"] = float(report[label]["precision"])
    summary[f"{label.lower()}_recall"] = float(report[label]["recall"])
    summary[f"{label.lower()}_f1"] = float(report[label]["f1-score"])
    summary[f"{label.lower()}_support"] = float(report[label]["support"])

print(json.dumps(summary, indent=2))

cm = confusion_matrix(YVA, y_pred, labels=LABELS)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print(cm_df)


In [ ]:
# 5. Plot confusion matrix (counts + row-normalised).
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title("SVM — Linear (LinearSVC) — Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title("SVM — Linear (LinearSVC) — Confusion Matrix (row-normalised)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()


## 6. Interpretation

- Reported scores are on the **validation** partition of the **primary** protocol (development data only).
- CV macro F1 was computed on the three fixed group-respecting development folds; TF-IDF/SVD were refit inside each fold.
- The locked test partition was not used for tuning or selection.
- Macro F1 treats all three classes equally; Risky precision/recall are reported alongside.
- Train macro F1 is shown for overfitting diagnosis only; it is not used for selection.
- Coefficients/rules are associations from case-derived annotations — not legal advice.
- A missing Risky prediction does not establish that a document is safe.

Saved results: `models/svm/linear/results/primary/metrics.csv`
Canonical comparison: `results/comparison/primary/comparison.csv`
